# Deep Learning Optimizers — Google Colab Notes

**Goal:** Understand what an optimizer does, how common optimizers differ, and how to use them in PyTorch.

## 1. What is an optimizer?
A neural network makes predictions, a **loss function** measures prediction error, and **backpropagation** computes gradients. The optimizer uses those gradients to update model parameters and reduce the loss.

Typical training loop:
1. Forward pass → predictions
2. Compute loss
3. `optimizer.zero_grad()` → clear old gradients
4. `loss.backward()` → calculate gradients
5. `optimizer.step()` → update parameters

A simplified update is: \(\theta_{t+1}=\theta_t-\eta\nabla_\theta L(\theta_t)\), where \(\theta\) represents parameters and \(\eta\) is the learning rate.

## 2. Learning rate
The learning rate controls the size of each parameter update.
- **Too high:** training can jump around or diverge.
- **Too low:** learning can be very slow.
- **A sensible starting point:** depends on the model and optimizer; test a few values rather than assuming one value always works.

## 3. Common optimizers

### SGD (Stochastic Gradient Descent)
Uses the gradient to update parameters. It is simple and can generalize well, but may need careful learning-rate tuning.

### SGD + Momentum
Accumulates a moving direction from past gradients, often reducing noisy or slow progress.

### Adagrad
Adapts the learning rate per parameter. It can be useful for sparse features, but its effective learning rate may become very small over time.

### RMSprop
Uses a moving average of squared gradients to adapt update sizes. Often useful for recurrent or non-stationary problems.

### Adam
Combines momentum-like averaging of gradients with adaptive scaling based on squared gradients. A common baseline for many deep-learning tasks.

### AdamW
Separates weight decay from Adam's gradient-based update. Often a good choice when using weight decay for regularization, especially in modern architectures.

## 4. Quick comparison

| Optimizer | Main idea | Often a reasonable use case | Watch out for |
|---|---|---|---|
| SGD | Gradient-based update | Strong baseline; vision models with tuned schedules | Learning-rate tuning matters |
| SGD + Momentum | Smooths update direction | Faster, steadier SGD training | Momentum and learning rate interact |
| Adagrad | Per-parameter adaptive rate | Sparse features | Rate can shrink too much |
| RMSprop | Moving average of squared gradients | Some sequence/non-stationary tasks | Needs tuning |
| Adam | Adaptive rate + gradient momentum | General-purpose starting baseline | May not always generalize best |
| AdamW | Adam with decoupled weight decay | Many modern neural networks | Tune weight decay and learning rate |

**Important:** No optimizer is universally best. Compare them using the same data split, model, training budget, and evaluation metrics.

In [ ]:
# 5. Setup (run this cell in Google Colab)
import torch
import torch.nn as nn
import torch.optim as optim

print('PyTorch:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 6. Create a small model
This example uses Fashion-MNIST so you can test optimizer behavior on a real classification task.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_dataset = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

model = FashionMLP().to(device)
criterion = nn.CrossEntropyLoss()
print(model)


## 7. Choose an optimizer
Uncomment **one** optimizer at a time. Start with Adam or AdamW, then compare with SGD + Momentum.

In [ ]:
learning_rate = 1e-3

# Option A: SGD
# optimizer = optim.SGD(model.parameters(), lr=0.01)

# Option B: SGD + Momentum
# optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Option C: Adagrad
# optimizer = optim.Adagrad(model.parameters(), lr=0.01)

# Option D: RMSprop
# optimizer = optim.RMSprop(model.parameters(), lr=1e-3)

# Option E: Adam
# optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Option F: AdamW (default choice for this notebook)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)


## 8. Train and evaluate
`CrossEntropyLoss` expects raw logits, so do not add a softmax layer to this model's final layer.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

epochs = 3  # Increase to 5–10 for a more meaningful experiment
for epoch in range(1, epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    test_loss, test_acc = run_epoch(model, test_loader, criterion)
    print(f'Epoch {epoch}: train loss={train_loss:.4f}, train acc={train_acc:.3%} | '
          f'test loss={test_loss:.4f}, test acc={test_acc:.3%}')


## 9. Common mistakes
- Forgetting `optimizer.zero_grad()` causes gradients to accumulate across batches.
- Forgetting `loss.backward()` means gradients are not computed.
- Forgetting `optimizer.step()` means parameters are not updated.
- Using an excessively high learning rate can make loss unstable.
- Comparing optimizers with different model initialization or training budgets can make the comparison unfair.
- Do not use the test set to repeatedly tune choices. Prefer a validation set for tuning and reserve the test set for final evaluation.

## 10. Mini experiment
1. Run the notebook with AdamW.
2. Restart the runtime and run with SGD + Momentum (`lr=0.01`, `momentum=0.9`).
3. Record training loss, validation/test accuracy, and training time.
4. Repeat with the same random seed and same epoch count for a fairer comparison.
5. Try learning rates such as `1e-4`, `1e-3`, and `1e-2` where appropriate.

### Key takeaway
An optimizer determines **how model parameters are updated** from gradients. Adam/AdamW are useful starting points, while SGD + Momentum is an important baseline. The best choice is determined by controlled experiments—not by accuracy on one run alone.